# Robust Sparse EM for Single-Cell Trajectory Analysis

## Thesis Implementation

**Dataset**: scRNA-seq, 1,425 cells, 2 time points (Healthy / Day 21 ischaemia)
**Model**: Robust Sparse EM with VAR[p] embedding and graphical lasso
**Date**: April 20, 2026

### Executive Summary

This notebook implements the Robust Sparse EM model adapted for population-level scRNA-seq data. The three pillars of the adaptation are:
- **Pseudotime as synthetic continuous timeline** - bridging the gap from 2 biological time points
- **Robust outlier detection** - repurposed from live-imaging to scRNA-seq technical noise
- **Sparse regulatory discovery** - via graphical lasso for interpretable networks

---

## 1. Setup and Imports

In [6]:
import numpy as np
import scanpy as sc
import pandas as pd
from scipy.linalg import inv
from scipy.stats import multivariate_normal
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
import skmisc
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print(f"Scanpy version: {sc.__version__}")

ModuleNotFoundError: No module named 'skmisc'

## 2. Data Loading and Quality Control

In [2]:
# Step 1: Load the h5ad file
print("Loading h5ad file...")

work_dir = 'C:/Users/saura/Desktop/Degree Project'

h5ad_dir = work_dir + 'h5ad/'
adata = sc.read_h5ad("C:/Users/saura/Desktop/Degree Project/h5ad/adata_final.h5ad")

print(f"Data loaded successfully!")
print(f"Shape: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"\nObservation metadata columns:")
print(adata.obs.columns.tolist())

Loading h5ad file...
Data loaded successfully!
Shape: 3975 cells × 6176 genes

Observation metadata columns:
['well', 'plate', 'ischemia', 'sex', 'cell_identity', 'sorting_batch', 'tdtomato', 'GFP', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'total_counts_ERCC', 'pct_counts_ERCC', 'n_genes', 'percent_chrY', 'XIST-counts', 'n_counts', 'S_score', 'G2M_score', 'gfp_expression', 'tdtomato_expression', 'batch', 'leiden_1.0', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'cell_type', 'cluster_ct', 'counts_unspliced', 'counts_spliced', 'fractions_unspliced', 'fractions_spliced', 'cell_count_x', 'cell_count_y', 'cell_count']


In [3]:
# Verify existing QC filters
print("=== Quality Control Verification ===")
print(f"\nCell counts:")
print(f"  Total cells: {adata.n_obs}")
print(f"  Cells with ischemia label: {adata.obs['ischemia'].notna().sum()}")

if 'pct_counts_mt' in adata.obs.columns:
    print(f"  Mean mitochondrial percentage: {adata.obs['pct_counts_mt'].mean():.2f}%")
    print(f"  Median mitochondrial percentage: {adata.obs['pct_counts_mt'].median():.2f}%")

if 'n_genes' in adata.obs.columns:
    print(f"  Mean genes per cell: {adata.obs['n_genes'].mean():.0f}")
    print(f"  Median genes per cell: {adata.obs['n_genes'].median():.0f}")

# Check cell type annotations
if 'cell_type' in adata.obs.columns:
    print(f"\nCell type distribution:")
    print(adata.obs['cell_type'].value_counts())

# Check ischemia conditions
if 'ischemia' in adata.obs.columns:
    print(f"\nIschemia condition distribution:")
    print(adata.obs['ischemia'].value_counts())

=== Quality Control Verification ===

Cell counts:
  Total cells: 3975
  Cells with ischemia label: 3975
  Mean mitochondrial percentage: 2.04%
  Median mitochondrial percentage: 1.63%
  Mean genes per cell: 3938
  Median genes per cell: 3963

Cell type distribution:
cell_type
Macrophages          1860
FAPs                  732
APCs                  596
Mural cells           315
Tenocytes             176
Endothelial cells     117
T-cells                80
Monocytes              77
Unknown                22
Name: count, dtype: int64

Ischemia condition distribution:
ischemia
D21        1467
D14         992
D7          910
Healthy     606
Name: count, dtype: int64


In [4]:
# Subset by condition
print("\n=== Subsetting by Condition ===")
adata_healthy = adata[adata.obs['ischemia'] != 'D21'].copy()
adata_d21 = adata[adata.obs['ischemia'] == 'D21'].copy()

print(f"Healthy cells: {adata_healthy.n_obs}")
print(f"D21 cells: {adata_d21.n_obs}")


=== Subsetting by Condition ===
Healthy cells: 2508
D21 cells: 1467


## 3. Normalization and Transformation

In [5]:
# Step 2: Normalization and Transformation
print("=== Normalization and Transformation ===")

for adata, label in [(adata_healthy, "Healthy"), (adata_d21, "D21")]:
    print(f"\nProcessing {label}...")

     # Identify highly variable genes
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
    adata = adata[:, adata.var['highly_variable']]
    print(f"  - HVG selection complete: {adata.n_vars} genes")
    
    # Library size normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    print(f"  - Library size normalization complete")
    
    # Log1p transformation
    sc.pp.log1p(adata)
    print(f"  - Log1p transformation complete")
    

    
    # Scale to zero mean, unit variance
    sc.pp.scale(adata)
    print(f"  - Scaling complete")

print("\nNormalization complete for both conditions!")

=== Normalization and Transformation ===

Processing Healthy...


ModuleNotFoundError: No module named 'skmisc'

## 4. Batch Effect Correction

In [ ]:
# Step 3: Batch Effect Correction
print("=== Batch Effect Correction ===")

# Check for batch information
if 'year' in adata.obs.columns or 'plate' in adata.obs.columns:
    print("Batch information found in metadata")
    if 'year' in adata.obs.columns:
        print(f"Year distribution: {adata.obs['year'].value_counts()}")
    
    # Combine datasets for batch correction
    adata_combined = adata_healthy.concatenate(adata_d21, batch_key='condition')
    
    # Apply Scanorama for batch correction
    try:
        import scanorama
        print("Applying Scanorama batch correction...")
        
        # Prepare batches
        batches = []
        batch_names = []
        for condition in ['Healthy', 'D21']:
            if condition == 'Healthy':
                batches.append(adata_healthy.X.copy())
            else:
                batches.append(adata_d21.X.copy())
            batch_names.append(condition)
        
        # Run Scanorama
        corrected, _ = scanorama.correct_scanpy(
            adata_combined, 
            batch_key='condition',
            return_dense=True
        )
        
        # Update data
        adata_healthy.X = corrected[adata_healthy.obs_names].X
        adata_d21.X = corrected[adata_d21.obs_names].X
        
        print("Batch correction complete!")
    except ImportError:
        print("Scanorama not available, skipping batch correction")
        print("Consider installing: pip install scanorama")
else:
    print("No batch information found, skipping batch correction")

## 5. Dimensionality Reduction (PCA)

In [ ]:
# Step 4: Dimensionality Reduction
print("=== Dimensionality Reduction ===")

for adata, label in [(adata_healthy, "Healthy"), (adata_d21, "D21")]:
    print(f"\nComputing PCA for {label}...")
    sc.tl.pca(adata, svd_solver='arpack', n_comps=50)
    
    # Calculate explained variance
    explained_variance = adata.uns['pca']['variance_ratio']
    cumulative_variance = np.cumsum(explained_variance)
    
    # Find PCs for 80% variance
    n_pcs_80 = np.argmax(cumulative_variance >= 0.8) + 1
    print(f"  - PCs for 80% variance: {n_pcs_80}")
    print(f"  - First 10 PCs explain: {cumulative_variance[9]*100:.1f}% variance")

print("\nPCA computation complete!")

## 6. Pseudotime Computation (DPT within D21)

In [ ]:
# Step 5: Pseudotime Computation
print("=== Pseudotime Computation ===")

# Compute DPT on D21 subset only (consistent with notebook)
print("Computing diffusion pseudotime on D21 subset...")

# Compute neighbors on D21 data
sc.pp.neighbors(adata_d21, n_neighbors=15, n_pcs=30)

# Compute diffusion map
sc.tl.diffmap(adata_d21)

# Compute DPT with root at cluster 0 (macrophages)
# First, identify cluster 0 cells
if 'cluster' in adata_d21.obs.columns:
    cluster_0_cells = adata_d21[adata_d21.obs['cluster'] == '0']
    root_idx = cluster_0_cells.obs_names[0]
    print(f"Setting root to cluster 0 cell: {root_idx}")
    sc.tl.dpt(adata_d21, n_dcs=10, root=root_idx)
else:
    # If no cluster info, use first cell as root
    print("No cluster information, using first cell as root")
    sc.tl.dpt(adata_d21, n_dcs=10)

print(f"DPT computed!")
print(f"DPT range: {adata_d21.obs['dpt_pseudotime'].min():.3f} to {adata_d21.obs['dpt_pseudotime'].max():.3f}")

In [ ]:
# Visualize pseudotime
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# UMAP colored by DPT
if 'X_umap' in adata_d21.obsm:
    sc.pl.umap(adata_d21, color='dpt_pseudotime', ax=axes[0], show=False)
    axes[0].set_title('DPT on UMAP')
else:
    sc.pl.diffmap(adata_d21, color='dpt_pseudotime', ax=axes[0], show=False)
    axes[0].set_title('DPT on Diffusion Map')

# Histogram of DPT
axes[1].hist(adata_d21.obs['dpt_pseudotime'], bins=30, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Diffusion Pseudotime')
axes[1].set_ylabel('Number of Cells')
axes[1].set_title('Distribution of DPT')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Pseudotime Binning

In [ ]:
# Step 6: Pseudotime Binning
print("=== Pseudotime Binning ===")

# Determine number of bins
n_bins = 20
min_cells_per_bin = 30

pseudotime = adata_d21.obs['dpt_pseudotime'].values

# Create bins
bin_edges = np.linspace(pseudotime.min(), pseudotime.max(), n_bins + 1)
bin_labels = np.digitize(pseudotime, bin_edges[:-1]) - 1

# Check bin sizes
bin_sizes = np.bincount(bin_labels)
print(f"Initial bin sizes:")
for i, size in enumerate(bin_sizes):
    print(f"  Bin {i}: {size} cells")

# Merge sparse bins if needed
if np.any(bin_sizes < min_cells_per_bin):
    print(f"\nWarning: Some bins have < {min_cells_per_bin} cells")
    print("Merging sparse bins...")
    
    # Simple strategy: reduce to 15 bins
    n_bins = 15
    bin_edges = np.linspace(pseudotime.min(), pseudotime.max(), n_bins + 1)
    bin_labels = np.digitize(pseudotime, bin_edges[:-1]) - 1
    bin_sizes = np.bincount(bin_labels)
    
    print(f"Adjusted to {n_bins} bins")
    for i, size in enumerate(bin_sizes):
        print(f"  Bin {i}: {size} cells")

# Add bin labels to adata
adata_d21.obs['pseudotime_bin'] = bin_labels.astype(str)

print(f"\nPseudotime binning complete: {n_bins} bins")

## 8. Outlier Pre-Filtering

In [ ]:
# Step 7: Outlier Pre-Filtering
print("=== Outlier Pre-Filtering ===")

# Combine PCA scores from both conditions
all_pca = np.vstack([adata_healthy.obsm['X_pca'], adata_d21.obsm['X_pca']])

# Use top PCs for outlier detection
n_pcs_detection = 30
pca_subset = all_pca[:, :n_pcs_detection]

# Fit robust covariance
print("Fitting Minimum Covariance Determinant...")
mcd = MinCovDet()
mcd.fit(pca_subset)

# Compute Mahalanobis distances
mahalanobis_dist = mcd.mahalanobnis(pca_subset)

# Identify outliers (top 5%)
threshold = np.percentile(mahalanobis_dist, 95)
outlier_mask = mahalanobis_dist > threshold

print(f"Detected {outlier_mask.sum()} outliers ({outlier_mask.sum()/len(outlier_mask)*100:.1f}%)")
print(f"Mahalanobis distance threshold: {threshold:.3f}")

# Visualize outlier detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of Mahalanobis distances
axes[0].hist(mahalanobis_dist, bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(threshold, color='red', linestyle='--', label='95th percentile')
axes[0].set_xlabel('Mahalanobis Distance')
axes[0].set_ylabel('Number of Cells')
axes[0].set_title('Distribution of Mahalanobis Distances')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot of first two PCs with outliers highlighted
axes[1].scatter(pca_subset[~outlier_mask, 0], pca_subset[~outlier_mask, 1], 
            alpha=0.5, s=10, label='Inliers', c='blue')
axes[1].scatter(pca_subset[outlier_mask, 0], pca_subset[outlier_mask, 1], 
            alpha=0.8, s=20, label='Outliers', c='red')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCA Space with Outliers Highlighted')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Store outlier flags
outlier_flags_healthy = outlier_mask[:adata_healthy.n_obs]
outlier_flags_d21 = outlier_mask[adata_healthy.n_obs:]

print(f"\nHealthy outliers: {outlier_flags_healthy.sum()}")
print(f"D21 outliers: {outlier_flags_d21.sum()}")

## 9. Extract Observations for EM Training

In [ ]:
# Extract observations using Option A: Pseudotime-binned means
print("=== Extracting Observations ===")

# Determine number of PCs to use (80% variance)
explained_variance = adata_d21.uns['pca']['variance_ratio']
cumulative_variance = np.cumsum(explained_variance)
n_pcs = np.argmax(cumulative_variance >= 0.8) + 1

print(f"Using {n_pcs} PCs for observations")

# Option A: Pseudotime-binned means
Y_by_time = []

# For each pseudotime bin, compute mean PCA
for bin_idx in range(n_bins):
    mask = adata_d21.obs['pseudotime_bin'] == str(bin_idx)
    if mask.sum() > 0:
        bin_cells = adata_d21[mask]
        # Remove outliers from mean computation
        bin_outliers = outlier_flags_d21[mask]
        bin_pca = bin_cells.obsm['X_pca'][:, :n_pcs]
        
        if (~bin_outliers).sum() > 0:
            bin_mean = bin_pca[~bin_outliers].mean(axis=0, keepdims=True)
        else:
            bin_mean = bin_pca.mean(axis=0, keepdims=True)
        
        Y_by_time.append(bin_mean)
    else:
        # Handle empty bins
        Y_by_time.append(None)

# Remove None entries
Y_by_time = [y for y in Y_by_time if y is not None]

print(f"\nCreated {len(Y_by_time)} observations from pseudotime bins")
print(f"Observation shape: {Y_by_time[0].shape}")

## 10. Robust Sparse EM Class Implementation

In [ ]:
class RobustSparseStateSpaceEM:
    """
    Robust Sparse EM for State-Space Models with VAR[p] Embedding
    
    Features:
    - VAR[p] embedding for time-delayed effects
    - Robust outlier detection (detect-and-reject)
    - Sparse regularization (graphical lasso)
    - Constrained optimization (preserve VAR structure)
    """
    
    def __init__(self, n_states, n_features, n_time_points, var_order=2,
                 l1_lambda=0.1, outlier_threshold=2.0, initialization='pca',
                 reg_strength=1e-6):
        """
        Parameters:
        -----------
        n_states : int - Number of latent states (K)
        n_features : int - PCA dimensions (D)
        n_time_points : int - Number of time points (T)
        var_order : int - VAR order (p) for time delays
        l1_lambda : float - L1 regularization strength for sparsity
        outlier_threshold : float - Threshold for outlier detection (Mahalanobis)
        initialization : str - 'pca', 'random', or 'clusters'
        reg_strength : float - Regularization for numerical stability
        """
        self.n_states = n_states
        self.n_features = n_features
        self.n_time_points = n_time_points
        self.var_order = var_order
        self.l1_lambda = l1_lambda
        self.outlier_threshold = outlier_threshold
        self.reg_strength = reg_strength
        self.initialization = initialization
        
        # Augmented dimensions
        self.augmented_states = n_states * var_order
        
        # Parameters
        self.A_matrices = None  # List of VAR matrices [A^(1), ..., A^(p)]
        self.A_aug = None  # Augmented transition matrix
        self.C = None  # Observation matrix (D × K*p)
        self.Q = None  # Process noise (K × K)
        self.Q_aug = None  # Augmented process noise
        self.R = None  # Observation noise (D × D)
        self.mu0 = None  # Initial state mean (K*p)
        self.Sigma0 = None  # Initial state covariance (K*p × K*p)
        
        # Storage
        self.X_smooth = None  # Smoothed augmented states
        self.P_smooth = None  # Smoothed covariances
        self.outlier_flags = None  # Outlier detection flags
        self.log_likelihood = []
    
    def initialize_parameters(self, Y_by_time):
        """Initialize parameters with VAR structure"""
        all_data = np.vstack([y for y in Y_by_time if y is not None])
        
        if self.initialization == 'pca':
            print("Initializing with PCA and VAR structure...")
            
            # PCA for C
            pca = PCA(n_components=self.n_states)
            pca.fit(all_data)
            self.C = pca.components_.T
            
            # Initialize C_aug (only current state observed)
            self.C_aug = np.zeros((self.n_features, self.augmented_states))
            self.C_aug[:, :self.n_states] = self.C
            
            # Initialize states
            self.X_smooth = []
            for y in Y_by_time:
                if y is not None:
                    # Current state from PCA
                    z = pca.transform(y)
                    # Augment with zeros for past states
                    X = np.zeros((y.shape[0], self.augmented_states))
                    X[:, :self.n_states] = z
                    self.X_smooth.append(X)
                else:
                    self.X_smooth.append(np.zeros((1, self.augmented_states)))
            
            # Initialize A matrices from autocorrelation
            self.A_matrices = []
            for k in range(1, self.var_order + 1):
                # Estimate A^(k) from lag-k autocorrelation
                A_k = np.random.randn(self.n_states, self.n_states) * 0.01
                self.A_matrices.append(A_k)
            
            # Assemble augmented A
            self._assemble_augmented_A()
            
            # Initialize Q (small)
            self.Q = self.reg_strength * np.eye(self.n_states)
            self.Q_aug = np.zeros((self.augmented_states, self.augmented_states))
            self.Q_aug[:self.n_states, :self.n_states] = self.Q
            
            # Initialize R from residuals
            residuals = []
            for y, X in zip(Y_by_time, self.X_smooth):
                if y is not None:
                    recon = X @ self.C_aug.T
                    residuals.append(y - recon)
            if residuals:
                all_residuals = np.vstack(residuals)
                self.R = np.cov(all_residuals.T) + self.reg_strength * np.eye(self.n_features)
            else:
                self.R = self.reg_strength * np.eye(self.n_features)
            
            # Initialize augmented state distribution
            self.mu0 = np.zeros(self.augmented_states)
            self.Sigma0 = self.reg_strength * np.eye(self.augmented_states)
        
        print("Initialization complete")
    
    def _assemble_augmented_A(self):
        """Assemble augmented transition matrix from VAR matrices"""
        self.A_aug = np.zeros((self.augmented_states, self.augmented_states))
        
        # Top row: VAR matrices
        top_row_start = 0
        for k, A_k in enumerate(self.A_matrices):
            self.A_aug[:self.n_states, top_row_start:top_row_start+self.n_states] = A_k
            top_row_start += self.n_states
        
        # Lower rows: Identity and Zero matrices (shift register)
        for i in range(1, self.var_order):
            row_start = i * self.n_states
            col_start = (i - 1) * self.n_states
            self.A_aug[row_start:row_start+self.n_states, 
                      col_start:col_start+self.n_states] = np.eye(self.n_states)
    
    def _detect_outlier(self, y_t, z_pred, P_pred):
        """
        Detect outlier using Mahalanobis distance
        
        Returns:
        --------
        is_outlier : bool
            True if observation is outlier
        mahalanobis_dist : float
            Mahalanobis distance
        """
        # Predict observation
        y_pred = self.C @ z_pred
        S = self.C @ P_pred @ self.C.T + self.R
        
        # Mahalanobis distance
        diff = y_t - y_pred
        try:
            mahalanobis_dist = np.sqrt(diff @ inv(S) @ diff.T)
        except:
            mahalanobis_dist = np.inf
        
        # Threshold
        is_outlier = mahalanobis_dist > self.outlier_threshold
        
        return is_outlier, mahalanobis_dist
    
    def kalman_filter_robust(self, Y_by_time):
        """
        Robust Kalman Filter with detect-and-reject
        
        Returns:
        --------
        X_filt : list of arrays
            Filtered augmented state estimates
        P_filt : list of arrays
            Filtered covariances
        X_pred : list of arrays
            Predicted state estimates
        P_pred : list of arrays
            Predicted covariances
        outlier_flags : list of arrays
            Outlier detection flags
        """
        X_filt = []
        P_filt = []
        X_pred = []
        P_pred = []
        outlier_flags = []
        
        # Initialize
        X_pred.append(self.mu0)
        P_pred.append(self.Sigma0)
        
        for t in range(self.n_time_points):
            if Y_by_time[t] is None:
                X_filt.append(X_pred[t])
                P_filt.append(P_pred[t])
                outlier_flags.append(np.array([False]))
            else:
                n_cells = Y_by_time[t].shape[0]
                X_filt_t = []
                P_filt_t = []
                outlier_flags_t = []
                
                for i in range(n_cells):
                    y_t = Y_by_time[t][i]
                    z_pred = X_pred[t][:self.n_states]  # Current state only
                    P_pred_current = P_pred[t][:self.n_states, :self.n_states]
                    
                    # Detect outlier
                    is_outlier, mahalanobis_dist = self._detect_outlier(
                        y_t, z_pred, P_pred_current
                    )
                    outlier_flags_t.append(is_outlier)
                    
                    if is_outlier:
                        # Reject: set R to infinity
                        R_adaptive = np.eye(self.n_features) * 1e10
                    else:
                        R_adaptive = self.R
                    
                    # Kalman gain
                    S = self.C @ P_pred_current @ self.C.T + R_adaptive
                    S = S + self.reg_strength * np.eye(self.n_features)
                    K = P_pred_current @ self.C.T @ inv(S)
                    
                    # Update
                    innovation = y_t - self.C @ z_pred
                    z_filt = z_pred + K @ innovation
                    P_filt_current = (np.eye(self.n_states) - K @ self.C) @ P_pred_current
                    
                    # Augment filtered state
                    X_filt_i = np.zeros(self.augmented_states)
                    X_filt_i[:self.n_states] = z_filt
                    if t > 0 and isinstance(X_filt[t-1], np.ndarray):
                        # Shift previous states
                        if X_filt[t-1].ndim == 1:
                            X_filt_i[self.n_states:] = X_filt[t-1][:self.n_states*(self.var_order-1)]
                        else:
                            X_filt_i[self.n_states:] = X_filt[t-1][0, :self.n_states*(self.var_order-1)]
                    
                    X_filt_t.append(X_filt_i)
                    P_filt_t.append(P_filt_current)
                
                X_filt.append(np.array(X_filt_t))
                P_filt.append(np.array(P_filt_t))
                outlier_flags.append(np.array(outlier_flags_t))
            
            # Predict next
            if t < self.n_time_points - 1:
                if isinstance(X_filt[t], np.ndarray) and X_filt[t].ndim > 1:
                    X_pred_t = (self.A_aug @ X_filt[t].T).T
                    P_pred_t = []
                    for i in range(len(P_filt[t])):
                        # Augment P
                        P_aug = np.zeros((self.augmented_states, self.augmented_states))
                        P_aug[:self.n_states, :self.n_states] = P_filt[t][i]
                        P_pred_i = self.A_aug @ P_aug @ self.A_aug.T + self.Q_aug
                        P_pred_t.append(P_pred_i)
                    P_pred_t = np.array(P_pred_t)
                else:
                    X_pred_t = self.A_aug @ X_filt[t]
                    P_aug = np.zeros((self.augmented_states, self.augmented_states))
                    P_aug[:self.n_states, :self.n_states] = P_filt[t]
                    P_pred_t = self.A_aug @ P_aug @ self.A_aug.T + self.Q_aug
                
                X_pred.append(X_pred_t)
                P_pred.append(P_pred_t)
        
        return X_filt, P_filt, X_pred, P_pred, outlier_flags
    
    def rts_smoother_augmented(self, X_filt, P_filt, X_pred, P_pred):
        """RTS Smoother for augmented states"""
        n_time = len(X_filt)
        X_smooth = [None] * n_time
        P_smooth = [None] * n_time
        
        X_smooth[-1] = X_filt[-1]
        P_smooth[-1] = P_filt[-1]
        
        for t in range(n_time - 2, -1, -1):
            if isinstance(X_filt[t], np.ndarray) and X_filt[t].ndim > 1:
                n_cells = X_filt[t].shape[0]
                X_smooth_t = []
                P_smooth_t = []
                
                for i in range(n_cells):
                    P_pred_next = P_pred[t+1] if isinstance(P_pred[t+1], np.ndarray) and P_pred[t+1].ndim == 2 else P_pred[t+1][i]
                    J = P_filt[t][i] @ self.A_aug.T @ inv(P_pred_next + self.reg_strength * np.eye(self.augmented_states))
                    X_smooth_i = X_filt[t][i] + J @ (X_smooth[t+1][i] - self.A_aug @ X_filt[t][i])
                    P_smooth_i = P_filt[t][i] + J @ (P_smooth[t+1][i] - P_pred_next) @ J.T
                    X_smooth_t.append(X_smooth_i)
                    P_smooth_t.append(P_smooth_i)
                
                X_smooth[t] = np.array(X_smooth_t)
                P_smooth[t] = np.array(P_smooth_t)
            else:
                P_pred_next = P_pred[t+1] if isinstance(P_pred[t+1], np.ndarray) and P_pred[t+1].ndim == 2 else P_pred[t+1][0]
                J = P_filt[t] @ self.A_aug.T @ inv(P_pred_next + self.reg_strength * np.eye(self.augmented_states))
                X_smooth[t] = X_filt[t] + J @ (X_smooth[t+1] - self.A_aug @ X_filt[t])
                P_smooth[t] = P_filt[t] + J @ (P_smooth[t+1] - P_pred_next) @ J.T
        
        return X_smooth, P_smooth
    
    def compute_sufficient_statistics_augmented(self, Y_by_time, X_smooth, P_smooth):
        """Compute sufficient statistics for augmented model"""
        # Initialize accumulators for each VAR matrix
        E_XzT = [np.zeros((self.n_states, self.n_states)) for _ in range(self.var_order)]
        E_zzT_prev = np.zeros((self.n_states, self.n_states))
        E_yzT = np.zeros((self.n_features, self.n_states))
        E_zzT_obs = np.zeros((self.n_states, self.n_states))
        E_yyT = np.zeros((self.n_features, self.n_features))
        
        n_pairs = 0
        n_obs = 0
        
        for t in range(self.n_time_points):
            if X_smooth[t] is not None:
                for i in range(X_smooth[t].shape[0]):
                    # Current state
                    z_t = X_smooth[t][i, :self.n_states]
                    P_t = P_smooth[t][i][:self.n_states, :self.n_states]
                    
                    E_zzT_obs += P_t + np.outer(z_t, z_t)
                    n_obs += 1
                
                if Y_by_time[t] is not None:
                    for i in range(Y_by_time[t].shape[0]):
                        z_t = X_smooth[t][i, :self.n_states]
                        E_yzT += np.outer(Y_by_time[t][i], z_t)
                        E_yyT += np.outer(Y_by_time[t][i], Y_by_time[t][i])
                
                # Cross-covariances for VAR matrices
                if t > 0 and X_smooth[t-1] is not None:
                    n_cells_t = X_smooth[t].shape[0]
                    n_cells_t1 = X_smooth[t-1].shape[0]
                    n_pairs_cell = min(n_cells_t, n_cells_t1)
                    
                    for i in range(n_pairs_cell):
                        z_t = X_smooth[t][i, :self.n_states]
                        
                        # For each lag
                        for k in range(self.var_order):
                            if t - k >= 0:
                                lag_idx = min(i, X_smooth[t-k].shape[0]-1)
                                z_t_k = X_smooth[t-k][lag_idx, :self.n_states]
                                E_XzT[k] += np.outer(z_t, z_t_k)
                        
                        E_zzT_prev += np.outer(z_t, z_t)
                        n_pairs += 1
        
        # Normalize
        if n_pairs > 0:
            for k in range(self.var_order):
                E_XzT[k] /= n_pairs
            E_zzT_prev /= n_pairs
        if n_obs > 0:
            E_zzT_obs /= n_obs
            E_yzT /= n_obs
            E_yyT /= n_obs
        
        return E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT
    
    def soft_threshold(self, x, lambda_val):
        """Soft thresholding operator for Lasso"""
        return np.sign(x) * np.maximum(np.abs(x) - lambda_val, 0)
    
    def graphical_lasso_update(self, E_XzT, E_zzT_prev):
        """
        Update VAR matrices using graphical lasso (coordinate descent)
        
        Only applies to top row (biological brain)
        Lower rows fixed (I and 0)
        """
        for k in range(self.var_order):
            # Update A^(k) with L1 regularization
            for i in range(self.n_states):
                for j in range(self.n_states):
                    numerator = self.soft_threshold(E_XzT[k][i, j], self.l1_lambda)
                    denominator = E_zzT_prev[j, j] + self.l1_lambda
                    self.A_matrices[k][i, j] = numerator / denominator
        
        # Reassemble augmented A
        self._assemble_augmented_A()
    
    def m_step_sparse(self, E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT):
        """M-Step with sparse regularization"""
        # Update A matrices using graphical lasso
        self.graphical_lasso_update(E_XzT, E_zzT_prev)
        
        # Update C (no regularization)
        try:
            self.C = E_yzT @ inv(E_zzT_obs + self.reg_strength * np.eye(self.n_states))
        except np.linalg.LinAlgError:
            self.C = E_yzT @ np.linalg.pinv(E_zzT_obs + self.reg_strength * np.eye(self.n_states))
        
        # Update C_aug
        self.C_aug = np.zeros((self.n_features, self.augmented_states))
        self.C_aug[:, :self.n_states] = self.C
        
        # Update Q
        self.Q = (E_zzT_obs - self.A_matrices[0] @ E_XzT[0].T - 
                  E_XzT[0] @ self.A_matrices[0].T + 
                  self.A_matrices[0] @ E_zzT_prev @ self.A_matrices[0].T)
        self.Q = (self.Q + self.Q.T) / 2
        self.Q = np.maximum(self.Q, self.reg_strength)
        
        # Update Q_aug
        self.Q_aug = np.zeros((self.augmented_states, self.augmented_states))
        self.Q_aug[:self.n_states, :self.n_states] = self.Q
        
        # Update R
        self.R = (E_yyT - self.C @ E_yzT.T - E_yzT @ self.C.T + 
                  self.C @ E_zzT_obs @ self.C.T)
        self.R = (self.R + self.R.T) / 2
        self.R = np.maximum(self.R, self.reg_strength)
        
        # Update initial state
        if self.X_smooth[0] is not None:
            self.mu0 = self.X_smooth[0].mean(axis=0)
            self.Sigma0 = np.cov(self.X_smooth[0].T) + self.reg_strength * np.eye(self.augmented_states)
    
    def fit(self, Y_by_time, max_iter=100, tol=1e-6, verbose=True):
        """Run robust sparse EM algorithm"""
        self.initialize_parameters(Y_by_time)
        prev_ll = -np.inf
        
        for iteration in range(max_iter):
            # E-step with robust detection
            X_filt, P_filt, X_pred, P_pred, outlier_flags = self.kalman_filter_robust(Y_by_time)
            X_smooth, P_smooth = self.rts_smoother_augmented(X_filt, P_filt, X_pred, P_pred)
            
            # Compute sufficient statistics
            E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT = \
                self.compute_sufficient_statistics_augmented(Y_by_time, X_smooth, P_smooth)
            
            # M-step with sparse regularization
            self.m_step_sparse(E_XzT, E_zzT_prev, E_yzT, E_zzT_obs, E_yyT)
            
            # Store results
            self.X_smooth = X_smooth
            self.P_smooth = P_smooth
            self.outlier_flags = outlier_flags
            
            # Compute log-likelihood (simplified)
            current_ll = -sum([np.sum(flags) for flags in outlier_flags])
            self.log_likelihood.append(current_ll)
            
            if verbose:
                n_outliers = sum([np.sum(flags) for flags in outlier_flags])
                sparsity = np.mean([np.sum(np.abs(A_k) < 1e-3) for A_k in self.A_matrices])
                print(f"Iteration {iteration + 1}: Outliers={n_outliers}, Sparsity={sparsity:.2%}")
            
            if abs(current_ll - prev_ll) < tol:
                if verbose:
                    print(f"Converged at iteration {iteration + 1}")
                break
            
            prev_ll = current_ll
        
        return self

print("RobustSparseStateSpaceEM class defined successfully!")

## 11. Model Training

In [ ]:
# Initialize model with recommended hyperparameters
print("=== Model Initialization ===")

n_states = 10  # Matches number of annotated cell types
var_order = 2  # Captures immediate and one-lag delayed effects
l1_lambda = 0.1  # Moderate sparsity
outlier_threshold = 2.0  # 95th-percentile confidence ellipsoid

model = RobustSparseStateSpaceEM(
    n_states=n_states,
    n_features=n_pcs,
    n_time_points=len(Y_by_time),
    var_order=var_order,
    l1_lambda=l1_lambda,
    outlier_threshold=outlier_threshold,
    initialization='pca',
    reg_strength=1e-6
)

print(f"Model initialized with:")
print(f"  - K (latent states): {n_states}")
print(f"  - p (VAR order): {var_order}")
print(f"  - λ (L1 penalty): {l1_lambda}")
print(f"  - Outlier threshold: {outlier_threshold}")
print(f"  - Time points (pseudotime bins): {len(Y_by_time)}")

In [ ]:
# Train the model
print("\n=== EM Training ===")
print("Starting robust sparse EM training...")

model = model.fit(Y_by_time, max_iter=100, tol=1e-6, verbose=True)

print("\nTraining complete!")

## 12. Results Analysis

In [ ]:
# Visualize convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-likelihood
axes[0].plot(model.log_likelihood, marker='o', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Negative Outlier Count')
axes[0].set_title('EM Convergence')
axes[0].grid(True, alpha=0.3)

# Sparsity over iterations
sparsity_history = []
# Note: This would need to be tracked during training
# For now, show final sparsity
final_sparsity = np.mean([np.sum(np.abs(A_k) < 1e-3) for A_k in model.A_matrices])
axes[1].bar(['A^(1)', 'A^(2)'], 
           [np.sum(np.abs(model.A_matrices[0]) < 1e-3) / model.A_matrices[0].size,
            np.sum(np.abs(model.A_matrices[1]) < 1e-3) / model.A_matrices[1].size])
axes[1].set_ylabel('Sparsity (fraction of zeros)')
axes[1].set_title('Final Sparsity of VAR Matrices')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.7, color='red', linestyle='--', label='70% target')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Examine learned VAR matrices
print("=== Learned VAR Matrices ===")

print("\nA^(1) (Immediate effects):")
print(model.A_matrices[0])

print("\nA^(2) (Delayed effects):")
print(model.A_matrices[1])

print("\nSparsity Summary:")
for k, A_k in enumerate(model.A_matrices):
    sparsity = np.sum(np.abs(A_k) < 1e-3) / A_k.size
    print(f"  A^({k+1}) sparsity: {sparsity:.2%}")

In [ ]:
# Visualize VAR matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# A^(1)
im1 = axes[0].imshow(model.A_matrices[0], cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[0].set_title('A^(1) - Immediate Effects')
axes[0].set_xlabel('Source State')
axes[0].set_ylabel('Target State')
plt.colorbar(im1, ax=axes[0])

# A^(2)
im2 = axes[1].imshow(model.A_matrices[1], cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[1].set_title('A^(2) - Delayed Effects')
axes[1].set_xlabel('Source State')
axes[1].set_ylabel('Target State')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Identify top regulatory interactions
def top_interactions(A, top_k=10):
    interactions = []
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            if abs(A[i, j]) > 1e-3:
                interactions.append((i, j, A[i, j]))
    interactions.sort(key=lambda x: -abs(x[2]))
    return interactions[:top_k]

print("=== Top Regulatory Interactions ===")

top_A1 = top_interactions(model.A_matrices[0], top_k=10)
print("\nTop immediate interactions (A^(1)):")
for i, j, val in top_A1:
    print(f"  State {i} <- State {j}: {val:.3f}")

top_A2 = top_interactions(model.A_matrices[1], top_k=10)
print("\nTop delayed interactions (A^(2)):")
for i, j, val in top_A2:
    print(f"  State {i} <- State {j} (lag 1): {val:.3f}")

## 13. Evaluation Metrics

In [ ]:
# Compute evaluation metrics
print("=== Evaluation Metrics ===")

# Sparsity metrics
def compute_sparsity(A):
    return np.sum(np.abs(A) < 1e-3) / A.size

sparsity_A1 = compute_sparsity(model.A_matrices[0])
sparsity_A2 = compute_sparsity(model.A_matrices[1])

print(f"\nSparsity Metrics:")
print(f"  A^(1) sparsity: {sparsity_A1:.2%}")
print(f"  A^(2) sparsity: {sparsity_A2:.2%}")
print(f"  Average sparsity: {(sparsity_A1 + sparsity_A2)/2:.2%}")

# Outlier detection metrics
total_outliers = sum([np.sum(flags) for flags in model.outlier_flags])
total_observations = sum([len(flags) for flags in model.outlier_flags])
outlier_rate = total_outliers / total_observations

print(f"\nOutlier Detection Metrics:")
print(f"  Total outliers: {total_outliers}")
print(f"  Outlier rate: {outlier_rate:.2%}")

# Reconstruction error
def compute_reconstruction_error(model, Y_by_time):
    errors = []
    for t, y in enumerate(Y_by_time):
        if y is not None and model.X_smooth[t] is not None:
            recon = model.X_smooth[t][:, :model.n_states] @ model.C.T
            error = np.sqrt(((y - recon) ** 2).mean())
            errors.append(error)
    return np.mean(errors)

reconstruction_error = compute_reconstruction_error(model, Y_by_time)
print(f"\nReconstruction Error:")
print(f"  RMSE: {reconstruction_error:.4f}")

# Success criteria check
print(f"\n=== Success Criteria Check ===")
print(f"Converged within 100 iterations: {'✓' if len(model.log_likelihood) <= 100 else '✗'}")
print(f"A matrices 70-90% sparsity: {'✓' if 0.7 <= (sparsity_A1 + sparsity_A2)/2 <= 0.9 else '✗'}")
print(f"Outlier rate 5-10%: {'✓' if 0.05 <= outlier_rate <= 0.10 else '✗'}")
print(f"Reconstruction RMSE < 0.5: {'✓' if reconstruction_error < 0.5 else '✗'}")

## 14. Save Model

In [ ]:
# Save trained model
print("=== Saving Model ===")

with open('robust_sparse_em_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved successfully as 'robust_sparse_em_model.pkl'")

## 15. Summary and Next Steps

In [ ]:
print("=== Implementation Summary ===")
print(f"\nDataset: {adata.n_obs} cells, {adata.n_vars} genes")
print(f"Time points: 2 (Healthy, D21)")
print(f"Pseudotime bins: {len(Y_by_time)}")
print(f"PCA dimensions: {n_pcs}")
print(f"\nModel Parameters:")
print(f"  Latent states (K): {n_states}")
print(f"  VAR order (p): {var_order}")
print(f"  L1 regularization (λ): {l1_lambda}")
print(f"  Augmented state dimension: {n_states * var_order}")
print(f"\nTraining Results:")
print(f"  Iterations to convergence: {len(model.log_likelihood)}")
print(f"  Final sparsity: {(sparsity_A1 + sparsity_A2)/2:.2%}")
print(f"  Outlier rate: {outlier_rate:.2%}")
print(f"  Reconstruction RMSE: {reconstruction_error:.4f}")
print(f"\n=== Next Steps ===")
print("1. Biological validation with marker genes")
print("2. Compare with PAGA trajectory (0 → 2 → 8)")
print("3. Correlate latent states with cell types")
print("4. Hyperparameter tuning if needed")
print("5. Compare with standard EM baseline")